In [ ]:
#use "../lib/nb.ml";;

# Evaluating an Exam Using Regular Expressions

This notebook shows how we can use regular expressions
to parse and process text. Our goal is to implement a program that can be used to evaluate the results of an exam. Assume the result of an exam is stored in the string `data` that is defined below:

In [1]:
let data = {|Class: Black Magic for Priests
          Group: TINF22AI1
          MaxPoints = 60

          Exercise:      1. 2. 3. 4. 5. 6.
          Jim Smith:     9 12 10  6  6  0
          John Slow:     4  4  2  0  -  -
          Susi Sorglos:  9 12 12  9  9  6
          1609922:       7  4 12  5  5  3
       |}

val data : string =
  "Class: Black Magic for Priests\n          Group: TINF22AI1\n          MaxPoints = 60\n\n          Exercise:      1. 2. 3. 4. 5. 6.\n          Jim Smith:     9 12 10  6  6  0\n          John Slow:     4  4  2  0  -  -\n          Susi Sorglos:  9 12 12  9  9  6\n          1609922:       7  4 12  5  5  3\n    "... (* string length 302; truncated *)


This data shows that there has been an exam with the subject <em style="color:blue">Black Magic for Priests</em>
in the group <em style="color:blue">TINF22AI1</em>. Furthermore, the equation
```
   MaxPoints = 60
```
shows that in order to achieve the best mark, <em style="color:blue">60</em> points would have been necessary.
    
There have been 6 different exercises in this exam and, in this small example, only four students took part, namely *Jim Smith*, *John Slow*, *Susi Sorglos*, and some student that is only represented by their matriculation number. Each of the rows describing the results of the students begins with the name (or matriculation number) of the student followed by the number of points that they have achieved in the different exercises. Our goal is to write a program that is able to compute the marks for all students.

## Imports

We will process the text using the module `Re` from the file `../lib/regex.ml`, which offers the functions of
Python's module `re` on top of the library PCRE2.

In [ ]:
load_lib "regex.ml"

## Auxiliary Functions

The function `mark max_points points` takes two arguments:
- `max_points` is the number of points that need to be achieved in order to get the best mark of $1.0$.
- `points` is the number of points achieved by the student whose mark is to be computed.

It is assumed that the relation between the mark of an exam and the number of points achieved in this exam is mostly linear and that a student who has achieved $50\%$ of `max_points` points will get the mark $4.0$, while a student who has achieved $100\%$ of `max_points` points will get the mark $1.0$. Therefore, the formula to calculate the grade is as follows:
$$\textrm{grade} = 7 - 6 \cdot \frac{\texttt{points}}{\texttt{max\_points}}$$
However, the worst mark is $5.0$. Therefore, if the mark would fall below that line, the `Float.min` function assures that it is less or equal to $5.0$. Furthermore, the resulting number is rounded to one digit.

**Type safety:** OCaml never converts an `int` into a `float` implicitly.  The function `float_of_int` converts
the integers `points` and `max_points` explicitly, and the operators `-.`, `*.`, and `/.` work on floating
point numbers only.  In Python, `points / max_points` is a floating point number while `points // max_points`
is an integer, and confusing these two operators is a common error.  OCaml has no function that rounds to a
given number of digits, so we format the number with one digit after the decimal point and convert the
result back into a `float`.  This rounds in the same way as Python's function `round`.

**Function `mark max_points points`**
- *Input:* `max_points` is the number of points that are needed for the best mark and `points` is the number of points that a student has achieved.
- *Output:* The mark of the student as a `float`, rounded to one digit after the decimal point.  The mark is at most $5.0$.

In [ ]:
let mark (max_points : int) (points : int) : float =
  let grade = 7. -. 6. *. float_of_int points /. float_of_int max_points in
  float_of_string (Printf.sprintf "%.1f" (Float.min 5.0 grade))

Let's test this function by plotting it.  OCaml has no counterpart of the Python library `matplotlib`, but
*JupyterLab* is able to display plots that are described in the language
[*Vega-Lite*](https://vega.github.io/vega-lite/).  A Vega-Lite plot is described by a *JSON* object that contains
the data and specifies how the data is to be displayed.  We build this JSON object with the library `yojson`,
which is loaded by `../lib/nb.ml`, and display it with the function `display_vegalite`.  A JSON value has
the type `Yojson.Safe.t`: for example, `` `Int 600 `` is a number, `` `String "line" `` is a string, and
`` `Assoc [...] `` is an object that is given as a list of pairs.  Since JSON values are typed, it is
impossible to produce a syntactically wrong JSON object.

In [ ]:
let vega_lite_line_plot ~(title : string) ~(xlabel : string) ~(ylabel : string)
                        (xs : int list) (ys : float list) : Yojson.Safe.t =
  let values = List.map2 (fun x y -> `Assoc [ ("x", `Int x); ("y", `Float y) ]) xs ys in
    (`Assoc [ ("$schema", `String "https://vega.github.io/schema/vega-lite/v5.json");
              ("title", `String title);
              ("width", `Int 600);
              ("height", `Int 400);
              ("data", `Assoc [ ("values", `List values) ]);
              ("mark", `Assoc [ ("type", `String "line"); ("point", `Bool true) ]);
              ("encoding",
               `Assoc [ ("x", `Assoc [ ("field", `String "x"); ("type", `String "quantitative");
                                      ("title", `String xlabel) ]);
                        ("y", `Assoc [ ("field", `String "y"); ("type", `String "quantitative");
                                      ("title", `String ylabel) ]) ]);
              ("config", `Assoc [ ("axis", `Assoc [ ("grid", `Bool true) ]) ]) ])

The next cell plots the function `mark` for values from `0` to `60` points.

In [ ]:
let max_points = 60
let points = List.init (max_points + 1) Fun.id
let grades = List.map (mark max_points) points
;;
display_vegalite
  (vega_lite_line_plot ~title:"Grade as a Function of Points (Max Points = 60)"
                       ~xlabel:"Points" ~ylabel:"Grade" points grades)

## Parsing the Data with Regular Expressions

Rather than tokenizing the entire string into a stream, we can process the input functionally line by line. We have two main tasks:
1. Extract the maximum number of points available from the document metadata.
2. Identify the student records, extract their scores, and compute their final marks.

### Extracting Maximum Points

To evaluate the exam, we must first isolate the maximum points definition. We can search the entire input string for the pattern `MaxPoints = [number]`. We use `\s*` to allow for arbitrary spacing around the equals sign, and capture the actual numeric value in a group so it can be extracted and cast to an integer.

**Function `get_max_points text`**
- *Input:* `text` is a string that contains the results of an exam.
- *Output:* The number of points that is specified by the line `MaxPoints = ...`.  The exception `Failure` is raised if there is no such line.

In [ ]:
let get_max_points (text : string) : int =
  match Re.search {|MaxPoints\s*=\s*([1-9][0-9]*)|} text with
  | Some m -> int_of_string (Re.group m 1)
  | None -> failwith "MaxPoints definition could not be found."

In [ ]:
get_max_points data

### Processing Student Records

Next, we need a function that evaluates each student.

A student line consists of an identifier followed by a colon. This identifier can either be a name (allowing upper/lower case letters, spaces, and hyphens) or a $7$-digit matriculation number. After the colon, a sequence of points is provided.

We can define two regular expressions for this task:
- `student` matches the beginning of a relevant line, capturing the student identifier in the first group and the remaining scores string in the second group.
- `pattern` is used to find all valid natural numbers (including zero) within the scores string, effectively ignoring hyphens that signify missed exercises.

Both regular expressions are compiled with `Re.compile`.  The functions `Re.Pattern.match_` and
`Re.Pattern.findall` correspond to the methods `match` and `findall` of a compiled Python regular expression.
`Re.Pattern.match_` returns a value of type `Re.match_ option`: Python's `match` returns either a match object or
`None`, and in OCaml the compiler forces us to handle the case `None`.

**Function `evaluate_students text max_points`**
- *Input:* `text` is a string that contains the results of an exam and `max_points` is the number of points that are needed for the best mark.
- *Output:* `()`.  The function prints the number of points and the mark of every student.

In [ ]:
let evaluate_students (text : string) (max_points : int) : unit =
  let student = Re.compile {|^\s*([A-Za-z \-]+|[0-9]{7}):\s*(.*)$|} in
  let pattern = Re.compile {|0|[1-9][0-9]*|} in
  List.iter
    (fun line ->
      let line = String.trim line in
      if line = ""
         || List.exists (fun prefix -> String.starts_with ~prefix line)
                        [ "Class:"; "Group:"; "MaxPoints"; "Exercise:" ]
      then ()
      else
        match Re.Pattern.match_ student line with
        | Some m ->
            let name   = String.trim (Re.group m 1) in
            let scores = Re.group m 2 in
            let points = List.fold_left (+) 0 (List.map int_of_string (Re.Pattern.findall pattern scores)) in
            let grade  = mark max_points points in
            Printf.printf "%s has %2d points and achieved the mark %.1f.\n%!" name points grade
        | None ->
            Printf.printf "Warning: could not parse the line '%s'.\n%!" line)
    (String.split_on_char '\n' (String.trim text))

### Running the Evaluation

Finally, we execute our parsing logic by passing in the original document string and the maximum points we extracted earlier.

In [ ]:
evaluate_students data max_points